# 04 — Unsupervised signature plots

This notebook:
- loads the fitted signature activities,
- creates overview heatmaps,
- merges activities with smoking annotation,
- creates smoking-group heatmaps,
- creates the SBS4 boxplot with individual points.


In [ ]:
%matplotlib inline

from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def show_image(path, width=1000):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Image not found: {path}")

def show_images(*paths, width=1000):
    for path in paths:
        show_image(path, width=width)

def get_sbs_columns(columns):
    cols = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(cols, key=lambda col: int(str(col)[3:]))

def normalize_rows(frame):
    frame = frame.copy()
    frame = frame.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = frame.sum(axis=1)
    frame = frame.loc[row_sums > 0].copy()
    return frame.div(frame.sum(axis=1), axis=0)

def extract_patient_id(value):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", str(value))
    return match.group(1) if match else np.nan

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"", "nan", "not reported", "unknown"}:
        return "Unknown"
    if "never" in text or "lifelong" in text or "non-smoker" in text or "nonsmoker" in text:
        return "Never"
    if "current reformed" in text or "reformed" in text or "former" in text:
        return "Former"
    if "current" in text:
        return "Current"
    return "Unknown"

def set_spaced_xticks(ax, labels, step=3):
    ax.set_xticks(np.arange(len(labels)) + 0.5)
    ax.set_xticklabels(labels, rotation=90)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % step != 0:
            label.set_visible(False)

    import scipy.stats as stats
    from scipy.stats import mannwhitneyu


## 1. Define paths


In [ ]:
activities_path = (
    PROJECT_ROOT
    / "results"
    / "LUAD_sig_output"
    / "Assignment_Solution"
    / "Activities"
    / "Assignment_Solution_Activities.txt"
)
clinical_path = PROJECT_ROOT / "data" / "clinical_exposure_merged.tsv"

overview_plot_dir = PROJECT_ROOT / "plots"
clinical_plot_dir = PROJECT_ROOT / "plots" / "supervised_heatmaps"

overview_plot_dir.mkdir(parents=True, exist_ok=True)
clinical_plot_dir.mkdir(parents=True, exist_ok=True)

print("Activities file:", activities_path)
print("Clinical file  :", clinical_path)


## 2. Load the fitted activities


In [ ]:
activities = pd.read_csv(activities_path, sep="\t", index_col=0)
sbs_cols = get_sbs_columns(activities.columns)

activity_matrix = activities[sbs_cols].copy()
activity_matrix = normalize_rows(activity_matrix)

print("Number of samples   :", activity_matrix.shape[0])
print("Number of signatures:", activity_matrix.shape[1])
print()

display(activity_matrix.iloc[:5, :10])

top_signatures = activity_matrix.sum(axis=0).sort_values(ascending=False).head(15)
display(
    top_signatures.rename("total_normalized_contribution")
    .reset_index()
    .rename(columns={"index": "signature"})
)


## 3. Create the grouped-sample heatmap


In [ ]:
group_size = 50
grouped_rows = []

for start in range(0, len(activity_matrix), group_size):
    stop = min(start + group_size, len(activity_matrix))
    group_name = f"Samples {start + 1}-{stop}"
    group_mean = activity_matrix.iloc[start:stop].mean(axis=0)
    grouped_rows.append(pd.Series(group_mean, name=group_name))

grouped_df = pd.DataFrame(grouped_rows)

plt.figure(figsize=(16, 8))
ax = sns.heatmap(
    grouped_df,
    cmap="Blues",
    linewidths=0.2,
    linecolor="gray",
    vmin=0,
    vmax=1,
)
set_spaced_xticks(ax, grouped_df.columns, step=3)
ax.set_title("Grouped sample signature contributions")
plt.tight_layout()

grouped_heatmap_path = overview_plot_dir / "heatmap_grouped_samples.png"
plt.savefig(grouped_heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", grouped_heatmap_path)


## 4. Create the top-50 heatmap


In [ ]:
top50_df = activity_matrix.copy()
top50_df["Total"] = top50_df.sum(axis=1)
top50_df = top50_df.sort_values("Total", ascending=False).drop(columns="Total").head(50)

plt.figure(figsize=(16, 10))
ax = sns.heatmap(
    top50_df,
    cmap="Blues",
    linewidths=0.2,
    linecolor="gray",
    vmin=0,
    vmax=1,
)
set_spaced_xticks(ax, top50_df.columns, step=3)
ax.set_title("Top 50 samples by total fitted activity")
plt.tight_layout()

top50_heatmap_path = overview_plot_dir / "heatmap_top50_samples.png"
plt.savefig(top50_heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", top50_heatmap_path)


## 5. Load the merged clinical table and prepare smoking groups


In [ ]:
clinical = pd.read_csv(clinical_path, sep="\t")

activity_matrix = activity_matrix.copy()
activity_matrix["Patient_ID"] = activity_matrix.index.map(extract_patient_id)

clinical = clinical.copy()
clinical["Patient_ID"] = clinical["cases.submitter_id"].map(extract_patient_id)

merged = pd.merge(
    activity_matrix,
    clinical,
    on="Patient_ID",
    how="inner",
    validate="many_to_one",
)

merged["Smoking_Category"] = merged["exposures.tobacco_smoking_status"].map(map_smoking_label)

print("Merged rows:", merged.shape[0])
print()
display(merged[["Patient_ID", "exposures.tobacco_smoking_status", "Smoking_Category"]].head())

smoking_counts = merged["Smoking_Category"].value_counts(dropna=False).rename_axis("Smoking_Category").reset_index(name="n_rows")
display(smoking_counts)


## 6. Aggregate to patient level


In [ ]:
patient_level = (
    merged
    .groupby("Patient_ID", as_index=False)
    .agg(
        {
            **{col: "mean" for col in sbs_cols},
            "Smoking_Category": "first",
            "exposures.pack_years_smoked": "first",
        }
    )
)

patient_level = patient_level[patient_level["Smoking_Category"].isin(["Never", "Former", "Current"])].copy()

patient_counts = patient_level["Smoking_Category"].value_counts().rename_axis("Smoking_Category").reset_index(name="n_patients")
display(patient_counts)


## 7. Create one heatmap for each smoking group


In [ ]:
for category in ["Never", "Former", "Current"]:
    category_df = patient_level[patient_level["Smoking_Category"] == category].copy()

    print()
    print(f"{category} patients:", len(category_df))

    if category_df.empty:
        continue

    heatmap_df = category_df[sbs_cols].T
    non_zero_cols = heatmap_df.columns[heatmap_df.sum(axis=0) > 0]
    heatmap_df = heatmap_df[non_zero_cols]

    plt.figure(figsize=(16, 8))
    sns.heatmap(
        heatmap_df,
        cmap="YlOrRd",
        linewidths=0.2,
        linecolor="gray",
        cbar_kws={"label": "Signature contribution"},
        yticklabels=True,
        xticklabels=False,
    )
    plt.title(f"Mutational signatures — {category} smokers (n={len(category_df)})")
    plt.ylabel("Signatures")
    plt.xlabel("Patients")
    plt.tight_layout()

    out_path = clinical_plot_dir / f"heatmap_{category}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved to:", out_path)


## 8. Create the SBS4 boxplot with individual points


In [ ]:
plot_df = patient_level.dropna(subset=["SBS4"]).copy()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=plot_df,
    x="Smoking_Category",
    y="SBS4",
    order=["Never", "Former", "Current"],
    width=0.6,
    fliersize=0,
    palette={"Never": "lightgreen", "Former": "orange", "Current": "red"},
)
sns.swarmplot(
    data=plot_df,
    x="Smoking_Category",
    y="SBS4",
    order=["Never", "Former", "Current"],
    size=3,
    alpha=0.7,
    color="black",
)
plt.title("SBS4 contribution by smoking category")
plt.xlabel("Smoking category")
plt.ylabel("SBS4 contribution")
plt.tight_layout()

sbs4_plot_path = overview_plot_dir / "SBS4_boxplot_swarmplot_clean.png"
plt.savefig(sbs4_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", sbs4_plot_path)

summary_sbs4 = (
    plot_df.groupby("Smoking_Category")["SBS4"]
    .agg(["count", "mean", "median", "std"])
    .round(4)
    .reset_index()
)
display(summary_sbs4)
